# ratings

En este cuadernillo se recoge el proceso de construcción del dataset de ratings y su partición en train, val y test.

In [1]:
import pandas as pd

## Partición en train, val y test
La partición se hace por usuario y de forma temporal, para evitar fugas de información (que el modelo aprenda con valoraciones futuras del mismo usuario).

Para cada usuario, sus valoraciones se ordenan por timestamp y se dividen en:
- Train: el primer 70% de las valoraciones (las más antiguas).
- Val: el siguiente 15% (del 70% al 85%).
- Test: el último 15% (las más recientes).

Las particiones de todos los usuarios se van acumulando en los dataframes train, val y test.

In [2]:
# PARTICIÓN TRAIN /VAL / TEST.
train = pd.DataFrame()
val   = pd.DataFrame()
test  = pd.DataFrame()

dfUser = pd.read_parquet("../data/02_processed/ratings_integrity.parquet")
dfUser = dfUser.sort_values(['userId', 'timestamp'])

def particion(ratingsUser):
    global train, val, test
    n       = len(ratingsUser)
    p_train = round(n * 0.70)
    p_val   = round(n * 0.85)

    trainPar = ratingsUser.head(p_train)
    valPar   = ratingsUser.iloc[p_train:p_val]
    testPar  = ratingsUser.tail(n - p_val)

    train = pd.concat([train, trainPar], ignore_index=True)
    val   = pd.concat([val,   valPar],   ignore_index=True)
    test  = pd.concat([test,  testPar],  ignore_index=True)

for i in dfUser['userId'].unique():
    particion(dfUser[dfUser['userId'] == i])

print(f"Train: {len(train)} ratings")
print(f"Val:   {len(val)} ratings")
print(f"Test:  {len(test)} ratings")

Train: 70342 ratings
Val:   15103 ratings
Test:  15060 ratings


Se guardan los tres conjuntos en formato parquet, en data/03_model_ready, listos para entrenar y evaluar el modelo.

In [3]:
train.to_parquet('../data/03_model_ready/ratings_train.parquet',index=False)
val.to_parquet('../data/03_model_ready/ratings_val.parquet',index=False)
test.to_parquet('../data/03_model_ready/ratings_test.parquet',index=False)